# Sesión 6 · Nube y Data Lakes — Tutorial de teoría + práctica

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

En esta sesión cerramos la capa de **almacenamiento** de la plataforma MLOps: dónde
viven los datos crudos, los artefactos y los modelos, y **cómo el Data Lake sostiene a
todos los protocolos que ya vimos** (streaming, gRPC, GraphQL y aprendizaje federado).

Trabajamos con **MinIO**, un almacenamiento de objetos **compatible con la API de Amazon
S3**. La idea clave: *el mismo código que corre contra MinIO en tu máquina corre igual
contra S3 en la nube*; solo cambian el endpoint y las credenciales.

### Qué vamos a hacer
1. Recordar la diferencia **Data Lake vs Data Warehouse**.
2. Levantar **MinIO** (Docker) y conectarnos con `boto3`.
3. Crear un bucket con **zonas** (`raw`, `staged`, `curated`, `models`).
4. Entrenar un modelo (el "modelo propio") y **subir dataset y modelo** al lake.
5. Registrar el experimento con **MLflow apuntando a MinIO/S3**.
6. **Servir** el modelo cargándolo desde el lake.
7. Ver el lake en acción con los **protocolos vistos**: streaming → lake, y serving por gRPC/GraphQL.

> Este notebook está pensado como **tutorial**: se lee y se ejecuta celda por celda.
> Corre de punta a punta con **uv**.


## 1. Data Lake vs Data Warehouse (recap breve)

| Criterio | **Data Lake** | **Data Warehouse** |
|---|---|---|
| Dato | Crudo, cualquier formato | Procesado, estructurado |
| Esquema | Al **leer** (schema-on-read) | Al **escribir** (schema-on-write) |
| Proceso | **ELT** (transformar después) | **ETL** (transformar antes) |
| Usuarios | Ciencia de datos, ML | Analítica de negocio, BI |
| Costo | Bajo (storage barato, desacoplado) | Alto (compute acoplado) |
| Mejor para | ML, exploración, dato crudo | Reportes, KPIs |

La ventaja del lake para ML: **se conserva el dato crudo** (se pueden crear features
nuevas más adelante), entra **cualquier formato**, y el almacenamiento es **barato y
elástico**. El precio es la **gobernanza**: sin disciplina, un lake se vuelve un *data
swamp*. Por eso organizamos el lake en **zonas**.


## 2. Requisitos y cómo levantar MinIO

El curso usa **[uv](https://docs.astral.sh/uv/)**. Instalar las dependencias de esta sesión:

```bash
uv add boto3 pandas pyarrow scikit-learn mlflow joblib
# (para usuarios de Poetry: poetry add boto3 pandas pyarrow scikit-learn mlflow joblib)
```

**Levantar MinIO con Docker** (deja el servicio escuchando en el puerto 9000, y la consola
web en el 9001):

```bash
docker run -d --name minio \
  -p 9000:9000 \
  -p 9001:9001 \
  -e MINIO_ROOT_USER=minio \
  -e MINIO_ROOT_PASSWORD=minio123 \
  cgr.dev/chainguard/minio:latest \
  server /data --console-address ":9001"
```

En **Windows (PowerShell)** el comando va en una sola línea, sin las barras `\`:

```powershell
docker run -d --name minio `
  -p 9000:9000 `
  -p 9001:9001 `
  -e MINIO_ROOT_USER=minio `
  -e MINIO_ROOT_PASSWORD=minio123 `
  cgr.dev/chainguard/minio:latest `
  server /data --console-address ":9001"
```

Con eso, la consola web queda en <http://localhost:9001> (usuario `minio`, clave `minio123`)
y la API S3 en <http://localhost:9000>. **Todo el código de abajo apunta a ese endpoint.**


In [12]:
# Configuración de conexión al Data Lake (MinIO / S3)
# Las credenciales van por variables de entorno: NUNCA hardcodear en producción.
import os

os.environ.setdefault("AWS_ACCESS_KEY_ID", "minio")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "minio123")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

ENDPOINT = "http://localhost:9000"   # MinIO local. En la nube: se quita para usar S3 real.
BUCKET = "datalake1"
print("Endpoint:", ENDPOINT, "| Bucket:", BUCKET)

Endpoint: http://localhost:9000 | Bucket: datalake1


In [13]:
import boto3
from botocore.client import Config

# Un único cliente S3. El MISMO objeto sirve para MinIO local y para S3 en la nube:
# para S3 real basta con no pasar endpoint_url y usar credenciales de AWS.
s3 = boto3.client(
    "s3",
    endpoint_url=ENDPOINT,
    config=Config(signature_version="s3v4"),
)
print("Cliente S3 listo.")

Cliente S3 listo.


## 3. Crear el bucket y las zonas del lake

Organizamos el lake en **zonas**, la mejor práctica para no caer en un *data swamp*:

- **`raw/`** (bronze): dato crudo tal cual llegó. Inmutable.
- **`staged/`** (silver): limpio, validado, normalizado (Parquet).
- **`curated/`** (gold): features y datasets listos para entrenar/servir.
- **`models/`**: artefactos de modelos, versionados (`models/v1/…`).

En S3 no existen "carpetas" reales: son **prefijos** en la clave (key) del objeto. Sirven
de partición lógica.


In [14]:
def crear_bucket(bucket):
    existentes = [b["Name"] for b in s3.list_buckets()["Buckets"]]
    if bucket in existentes:
        print(f"El bucket '{bucket}' ya existe.")
        return
    s3.create_bucket(Bucket=bucket)
    print(f"Bucket '{bucket}' creado.")

crear_bucket(BUCKET)

# Las zonas se materializan al subir el primer objeto con ese prefijo.
ZONAS = ["raw", "staged", "curated", "models"]
for z in ZONAS:
    s3.put_object(Bucket=BUCKET, Key=f"{z}/.keep", Body=b"")
print("Zonas inicializadas:", ZONAS)

Bucket 'datalake1' creado.
Zonas inicializadas: ['raw', 'staged', 'curated', 'models']


## 4. Entrenar un modelo y subir dataset + modelo al lake

Usamos un dataset pequeño de `scikit-learn` como *stand-in* del **modelo propio**. El flujo
es el mismo con cualquier modelo: entrenar → guardar el artefacto → subirlo al lake.


In [4]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

data = load_wine(as_frame=True)
df = data.frame                       # features + columna 'target'
X = df.drop(columns=["target"])
y = df["target"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

modelo = RandomForestClassifier(n_estimators=200, random_state=42)
modelo.fit(Xtr, ytr)
acc = accuracy_score(yte, modelo.predict(Xte))
print(f"Modelo entrenado. Accuracy de test: {acc:.3f}")

Modelo entrenado. Accuracy de test: 1.000


In [15]:
import io, joblib

# 4a. Subir el dataset CRUDO a la zona raw (CSV, tal cual).
buf_csv = io.StringIO(); df.to_csv(buf_csv, index=False)
s3.put_object(Bucket=BUCKET, Key="raw/wine/wine.csv",
              Body=buf_csv.getvalue().encode("utf-8"))

# 4b. Subir el dataset CURADO a la zona curated (Parquet, columnar y liviano).
buf_pq = io.BytesIO(); df.to_parquet(buf_pq, index=False)
s3.put_object(Bucket=BUCKET, Key="curated/wine/wine.parquet", Body=buf_pq.getvalue())

# 4c. Subir el MODELO a la zona models, VERSIONADO (models/v1/...).
buf_model = io.BytesIO(); joblib.dump(modelo, buf_model)
s3.put_object(Bucket=BUCKET, Key="models/v1/wine_rf.pkl", Body=buf_model.getvalue())

print("Subidos: raw/wine/wine.csv, curated/wine/wine.parquet, models/v1/wine_rf.pkl")

Subidos: raw/wine/wine.csv, curated/wine/wine.parquet, models/v1/wine_rf.pkl


In [16]:
# Listar el contenido del lake (el "árbol" de objetos).
def listar(bucket, prefix=""):
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    for obj in resp.get("Contents", []):
        if obj["Key"].endswith("/.keep"):
            continue
        print(f"  {obj['Key']:45s} {obj['Size']:>10,} bytes")

print(f"Contenido de s3://{BUCKET}/")
listar(BUCKET)

Contenido de s3://datalake1/
  curated/wine/wine.parquet                         16,937 bytes
  models/v1/wine_rf.pkl                            422,273 bytes
  raw/wine/wine.csv                                 12,440 bytes


## 5. MLflow apuntando a MinIO/S3

En una plataforma real, **MLflow** registra experimentos (parámetros, métricas) y **guarda
los artefactos en el lake**. Configuramos MLflow para que use MinIO como *artifact store*:

- El **tracking backend** (metadatos de las corridas) va a una base SQLite local.
- El **artifact store** apunta a `s3://datalake/mlflow`, es decir, **a nuestro lake**.
- La variable `MLFLOW_S3_ENDPOINT_URL` le dice a MLflow que hable con MinIO y no con AWS.


In [17]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = ENDPOINT   # MinIO como backend S3 de MLflow
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("sqlite:///mlflow.db")   # backend recomendado (no filestore)

# Experimento cuyos artefactos viven en el lake:
client = MlflowClient()
nombre_exp = "sesion6-datalake"
existente = client.get_experiment_by_name(nombre_exp)
if existente is None:
    exp_id = client.create_experiment(nombre_exp, artifact_location=f"s3://{BUCKET}/mlflow")
else:
    exp_id = existente.experiment_id
mlflow.set_experiment(experiment_id=exp_id)
print("Experimento MLflow listo (artefactos → lake).")

Experimento MLflow listo (artefactos → lake).


In [18]:
with mlflow.start_run(run_name="rf-wine") as run:
    mlflow.log_param("n_estimators", 200)
    mlflow.log_metric("accuracy", float(acc))
    # Guardar el modelo como artefacto: MLflow lo sube al lake (MinIO/S3).
    joblib.dump(modelo, "wine_rf.pkl")
    mlflow.log_artifact("wine_rf.pkl", artifact_path="model")
    run_id = run.info.run_id
print("Run registrado:", run_id)

# Verificar que el artefacto quedó en el lake:
print("\nArtefactos de MLflow en el lake:")
listar(BUCKET, prefix="mlflow")

Run registrado: 75ff90bd7aa74670bc6978bce19e8784

Artefactos de MLflow en el lake:


## 6. Servir el modelo cargándolo desde el lake

Este es el patrón de despliegue clave: **el modelo NO se hornea en la imagen del servicio**.
El servicio lo **descarga del lake al arrancar**. Actualizar el modelo = subir una versión
nueva al lake (`models/v2/…`) y reiniciar el servicio. Esto es exactamente lo que hace un
endpoint REST/gRPC en producción.


In [20]:
def cargar_modelo_desde_lake(bucket, key):
    # Descarga un modelo del lake y lo deja listo para predecir.
    obj = s3.get_object(Bucket=bucket, Key=key)
    return joblib.load(io.BytesIO(obj["Body"].read()))

# El "servicio" arranca y carga su modelo desde el lake:
modelo_servido = cargar_modelo_desde_lake(BUCKET, "models/v1/wine_rf.pkl")

# Simular una predicción como lo haría el endpoint:
muestra = Xte.iloc[[0]]
pred = modelo_servido.predict(muestra)[0]
print("Modelo cargado desde el lake. Predicción de ejemplo:", int(pred),
      "| clase real:", int(yte.iloc[0]))

Modelo cargado desde el lake. Predicción de ejemplo: 0 | clase real: 0


## 7. El Data Lake y los protocolos vistos

Todo lo que vimos en el bimestre **se apoya en el lake**. Lo mostramos con dos ejemplos
concretos.

### 7.1 Streaming → Data Lake
Un stream (Kafka) es **efímero**: los eventos tienen retención corta. El lake le da
**memoria durable**: el consumidor **persiste** cada lote de eventos como objeto,
**particionado por fecha** (`eventos/fecha=YYYY-MM-DD/…`). Así el evento que pasó por el
stream sigue disponible para reentrenar y auditar.


In [21]:
import json, datetime, random

# Simulamos un stream de eventos (como los de la Sesión 4) y los aterrizamos en el lake.
def evento_falso(i):
    fila = Xte.iloc[i % len(Xte)]
    return {"id": i, "features": fila.tolist(),
            "ts": datetime.datetime.utcnow().isoformat()}

hoy = datetime.date.today().isoformat()
buffer, parte = [], 0
for i in range(1200):                       # "consumo" del stream
    buffer.append(evento_falso(i))
    if len(buffer) >= 500:                   # al llenar el lote, se persiste al lake
        key = f"raw/eventos/fecha={hoy}/parte-{parte:03d}.parquet"
        cuerpo = pd.DataFrame(buffer).to_parquet(index=False)
        s3.put_object(Bucket=BUCKET, Key=key, Body=cuerpo)
        print("Persistido:", key, f"({len(buffer)} eventos)")
        buffer, parte = [], parte + 1

print("\nEl stream fluyó; el lake recuerda. Partición del día:")
listar(BUCKET, prefix=f"raw/eventos/fecha={hoy}")

Persistido: raw/eventos/fecha=2026-10-01/parte-000.parquet (500 eventos)
Persistido: raw/eventos/fecha=2026-10-01/parte-001.parquet (500 eventos)

El stream fluyó; el lake recuerda. Partición del día:
  raw/eventos/fecha=2026-10-01/parte-000.parquet     12,338 bytes
  raw/eventos/fecha=2026-10-01/parte-001.parquet     12,354 bytes


### 7.2 gRPC / GraphQL sobre el lake

- **gRPC**: el servidor de inferencia carga su modelo **desde el lake** (celda 6) y responde
  con baja latencia. Versionar el modelo = subir `models/vN/…` y reiniciar.
- **GraphQL**: cada *resolver* consulta el lake como **origen de verdad**. Por ejemplo, para
  devolver una predicción ya calculada, lee un objeto de la zona `curated`.

Simulamos el patrón del resolver de GraphQL: guardar una predicción en `curated` y leerla
después.


In [22]:
# El servicio calcula predicciones y las deja en la zona curated:
preds = modelo_servido.predict(Xte)
tabla_pred = pd.DataFrame({"id": range(len(preds)), "prediccion": preds})
s3.put_object(Bucket=BUCKET, Key="curated/preds/wine_v1.parquet",
              Body=tabla_pred.to_parquet(index=False))

# El resolver de GraphQL resuelve un campo leyendo del lake:
def resolver_prediccion(id_muestra):
    obj = s3.get_object(Bucket=BUCKET, Key="curated/preds/wine_v1.parquet")
    t = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    fila = t[t["id"] == id_muestra]
    return None if fila.empty else int(fila["prediccion"].iloc[0])

print("resolver_prediccion(5) →", resolver_prediccion(5))
print("El lake es el origen de verdad para gRPC (modelo) y GraphQL (features/predicciones).")

resolver_prediccion(5) → 0
El lake es el origen de verdad para gRPC (modelo) y GraphQL (features/predicciones).


## 8. Cierre

Recorrimos el rol del Data Lake en una plataforma MLOps:

- **Zonas** (`raw`/`staged`/`curated`/`models`) para no caer en un *data swamp*.
- **MinIO = S3 local**: el mismo `boto3` sirve para practicar y para producción.
- **MLflow** versiona experimentos y guarda artefactos **en el lake**.
- **Serving**: el modelo se **carga desde el lake**, no se hornea en la imagen.
- Todo lo visto converge en el lake: **streaming** aterriza eventos, **gRPC** sirve el
  modelo, **GraphQL** consulta features/predicciones, y en **federado** el modelo global
  se versiona en MinIO/MLflow.

➡️ **Mini-TP 6:** subir el artefacto del modelo propio al lake y servirlo cargándolo desde
allí. Starter en `clase6/Practica/mini_tp6_actividad.ipynb`.
